**Import libraries**

In [ ]:
import itertools
import json
from pathlib import Path
 
import joblib
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

**Config: where fitted objects and feature tables get saved**

In [ ]:
FEATURE_ARTIFACTS_DIR = Path("feature_engineering_artifacts")  # from Notebook 5
MODEL_ARTIFACTS_DIR = Path("model_artifacts")  # this notebook's own artifacts
MODEL_ARTIFACTS_DIR.mkdir(exist_ok=True)
 
TARGET = "is_late"
RANDOM_STATE = 42

**Read artifacts: train, validation, and test**


In [ ]:
train_df = pd.read_parquet("feature_table_train.parquet")
val_df = pd.read_parquet("feature_table_val.parquet")
test_df = pd.read_parquet("feature_table_test.parquet")
 
with open(FEATURE_ARTIFACTS_DIR / "final_feature_list.json") as f:
    feature_list = json.load(f)
 
X_train, y_train = train_df[feature_list], train_df[TARGET].astype(int)
X_val, y_val = val_df[feature_list], val_df[TARGET].astype(int)
X_test, y_test = test_df[feature_list], test_df[TARGET].astype(int)
 
print("Train:", X_train.shape, " positive rate:", round(y_train.mean(), 4))
print("Val:  ", X_val.shape, " positive rate:", round(y_val.mean(), 4))
print("Test: ", X_test.shape, " positive rate:", round(y_test.mean(), 4))
 
 
def evaluate(model, X, y, label=""):
    """Metric set that fits an imbalanced problem — not accuracy alone."""
    y_pred = model.predict(X)
    y_proba = model.predict_proba(X)[:, 1]
    return {
        "set": label,
        "f1": round(f1_score(y, y_pred), 4),
        "precision": round(precision_score(y, y_pred), 4),
        "recall": round(recall_score(y, y_pred), 4),
        "roc_auc": round(roc_auc_score(y, y_proba), 4),
        "pr_auc": round(average_precision_score(y, y_proba), 4),
    }
 
 
results_log = []  # every experiment appended here -> results summary artifact

**Baseline**

In [ ]:
baseline = DummyClassifier(strategy="stratified", random_state=RANDOM_STATE)
baseline.fit(X_train, y_train)
 
baseline_metrics = evaluate(baseline, X_val, y_val, label="baseline_dummy (val)")
results_log.append(baseline_metrics)
print("\nBaseline (Dummy, stratified) on validation:", baseline_metrics)

**Train Random Forest, tune on VALIDATION**

In [ ]:
param_grid = {
    "n_estimators": [200, 400],
    "max_depth": [None, 10, 20],
    "min_samples_leaf": [1, 5, 10],
}
 
grid_keys = list(param_grid.keys())
grid_values = list(param_grid.values())
 
best_f1 = -1
best_model = None
best_params = None
 
for combo in itertools.product(*grid_values):
    params = dict(zip(grid_keys, combo))
    rf = RandomForestClassifier(
        **params,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    rf.fit(X_train, y_train)
 
    metrics = evaluate(rf, X_val, y_val, label=f"rf {params} (val)")
    results_log.append(metrics)
    print(metrics)
 
    if metrics["f1"] > best_f1:
        best_f1 = metrics["f1"]
        best_model = rf
        best_params = params
 
print("\nBest params (by F1 on validation):", best_params)
print("Best validation F1:", best_f1)

final_model = best_model

**test the final model on the test set**

In [ ]:
test_metrics = evaluate(final_model, X_test, y_test, label="FINAL MODEL (test)")
results_log.append(test_metrics)
 
print("\n=== FINAL TEST RESULTS (touched once) ===")
print(test_metrics)
print("\nClassification report (test):")
print(classification_report(y_test, final_model.predict(X_test), digits=4))
print("Confusion matrix (test):")
print(confusion_matrix(y_test, final_model.predict(X_test)))

**Artifacts: trained model + results summary**

In [ ]:
joblib.dump(final_model, MODEL_ARTIFACTS_DIR / "final_model_random_forest.joblib")
 
results_summary = pd.DataFrame(results_log)
results_summary.to_csv(MODEL_ARTIFACTS_DIR / "results_summary.csv", index=False)
 
feature_importances = (
    pd.Series(final_model.feature_importances_, index=feature_list)
    .sort_values(ascending=False)
)
feature_importances.to_csv(MODEL_ARTIFACTS_DIR / "feature_importances.csv", header=["importance"])
 
with open(MODEL_ARTIFACTS_DIR / "final_model_config.json", "w") as f:
    json.dump(
        {
            "best_params": best_params,
            "class_weight": "balanced",
            "tuning_metric": "f1",
            "decision_threshold": 0.5,
            "final_test_metrics": test_metrics,
        },
        f,
        indent=2,
    )
 
print(f"\nSaved to '{MODEL_ARTIFACTS_DIR}/':")
for f_ in sorted(MODEL_ARTIFACTS_DIR.iterdir()):
    print(" -", f_.name)
 
results_summary